# Getting Started with the Sense HAT — Java 25 + Pi4J

Welcome! This is a self-contained, beginner-friendly tour of the Raspberry Pi **Sense HAT**. It doesn't depend on the other notebooks in this repo — no Spark, no Floci, no lakehouse. Just you, a Pi, and the sensor board.

**What you'll learn:**

1. What the Sense HAT is and what's on it
2. How to enable and check I2C on the Pi
3. What Pi4J is and how it talks to hardware
4. How to install dependencies for a Java notebook
5. How to read every sensor
6. How to draw on the LED matrix
7. How to react to the joystick
8. How to clean up properly

**Prerequisites:**

- A Raspberry Pi (any model with the 40-pin GPIO header)
- A Sense HAT plugged into the GPIO header
- Raspberry Pi OS, 64-bit
- Java 25 available inside the Jupyter environment
- The Jupyter Java (IJava) kernel

If you're running this notebook inside the Docker container from this repo, Java 25 and the kernel are already set up for you.


## 1. What is the Sense HAT?

The Sense HAT is an add-on board designed for the Raspberry Pi Foundation's [Astro Pi](https://astro-pi.org/) programme — the same hardware that flies on the International Space Station. It packs a lot of sensors into a small board:

- 8x8 RGB LED matrix (64 LEDs)
- 5-button mini joystick
- **HTS221** — humidity + temperature
- **LPS25H** — barometric pressure + temperature
- **LSM9DS1** — 9-DoF IMU (accelerometer + gyroscope + magnetometer)
- **TCS3400** — colour and light sensor (only on Sense HAT v2)

All of these chips talk to the Pi over a shared bus called **I2C**. The LED matrix is driven by an ATtiny microcontroller that also sits on the same bus. You don't have to worry about that layer — the driver takes care of it — but it explains why the very first thing we do is enable I2C.

If you want to peek at the driver source, it's here: [pi4j-drivers SenseHat.java](https://github.com/igfasouza/pi4j-drivers/blob/igfasouza/src/main/java/com/pi4j/drivers/hat/raspberry/SenseHat.java).


## 2. Enable and Check I2C

I2C is a bus protocol used to talk to sensors over just two wires (clock + data). The Sense HAT chips all live on the Pi's I2C bus, which shows up as `/dev/i2c-1`.

### 2.1. Enable I2C on the Pi (host, once)

Run this once on the Pi host (**not** inside Docker):

```bash
sudo raspi-config
# → Interface Options → I2C → Enable → Reboot
```

Or non-interactively:

```bash
sudo raspi-config nonint do_i2c 0
sudo reboot
```

### 2.2. Give the container access to I2C

If you're running this notebook inside Docker, the container needs to see `/dev/i2c-1`. In `docker-compose.yml`:

```yaml
devices:
  - "/dev/i2c-1:/dev/i2c-1"
privileged: true
```

For the LED matrix you may also want the framebuffer:

```yaml
devices:
  - "/dev/i2c-1:/dev/i2c-1"
  - "/dev/fb0:/dev/fb0"
```

### 2.3. Scan the I2C bus

The cell below runs `i2cdetect -y 1`. If the Sense HAT is wired up correctly, you should see something like:

```
     0  1  2  3  4  5  6  7  8  9  a  b  c  d  e  f
00:          -- -- -- -- -- -- -- -- -- -- -- -- --
10: -- -- -- -- -- -- -- -- -- -- -- 1b -- -- -- --
20: -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --
30: -- -- -- -- -- -- -- -- -- -- -- -- -- -- 3f --
40: -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --
50: -- -- -- -- -- -- -- -- -- -- -- -- 5c -- -- 5f
60: -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- --
70: -- -- -- -- -- -- --
```

The important addresses:

| Address | Chip     | Role                       |
|---------|----------|----------------------------|
| `0x1c`  | LSM9DS1  | Magnetometer               |
| `0x1b`  | LSM9DS1  | Accel + gyro (Sense HAT v1)|
| `0x29`  | TCS3400  | Colour sensor (v2)         |
| `0x46`  | ATtiny   | LED matrix + joystick      |
| `0x5c`  | LPS25H   | Pressure                   |
| `0x5f`  | HTS221   | Humidity                   |

Missing addresses usually mean the HAT isn't seated firmly or I2C isn't enabled.


In [ ]:
// Scan the I2C bus from inside the notebook. Requires `i2c-tools` (installed
// in the Dockerfile) and the container privileged / mapped to /dev/i2c-1.
Process proc = new ProcessBuilder("i2cdetect", "-y", "1")
    .redirectErrorStream(true)
    .start();

try (var reader = new java.io.BufferedReader(
        new java.io.InputStreamReader(proc.getInputStream()))) {
    reader.lines().forEach(System.out::println);
}

int exit = proc.waitFor();
System.out.println("(exit=" + exit + ")");


## 3. What is Pi4J?

[**Pi4J**](https://pi4j.com/) is a Java library for talking to the Raspberry Pi's GPIO, I2C, SPI, and other low-level peripherals. Its role is roughly the same as the popular Python `RPi.GPIO` + `smbus` + `sense-hat` combination — but for Java.

Pi4J is split into three layers:

1. **Core** (`pi4j-core`) — the API you program against: `Context`, `I2C`, `Spi`, `DigitalOutput`, …
2. **Providers/plugins** — the platform-specific implementation. On modern 64-bit Raspberry Pi OS we use the **FFM (Foreign Function & Memory) plugin**, which talks directly to the kernel via `libc` — no root, no sysfs, no wrappers.
3. **Drivers** (`pi4j-drivers`) — high-level, opinionated wrappers for popular boards and chips. The Sense HAT lives here as `com.pi4j.drivers.hat.raspberry.SenseHat`.

The mental model:

```
your Java code
   └── com.pi4j.drivers.hat.raspberry.SenseHat   ← the friendly API
         └── pi4j-core (Context, I2C, ...)       ← generic Pi4J
               └── pi4j-plugin-ffm               ← platform backend
                     └── /dev/i2c-1 (kernel)     ← the actual bus
```

You only need to interact with the top layer for this notebook.


## 4. Install Dependencies

The Jupyter Java kernel (IJava) supports Maven-style dependencies with the `%maven` magic. Three artifacts are enough for the whole tour:

- `com.pi4j:pi4j-core` — the core API
- `com.pi4j:pi4j-plugin-ffm` — the FFM backend
- `com.pi4j:pi4j-drivers` — the Sense HAT driver

The `pi4j-drivers` artifact is built for Java 25, so the Java kernel needs Java 25 too.


In [ ]:
%maven com.pi4j:pi4j-core:5.0.0-SNAPSHOT
%maven com.pi4j:pi4j-plugin-ffm:5.0.0-SNAPSHOT
%maven com.pi4j:pi4j-drivers:1.1.0


### 4.1. Confirm the Java version

If this prints anything other than `25.x`, the Sense HAT driver will fail to load. Fix `JAVA_HOME` in your Dockerfile / environment before continuing.


In [ ]:
System.out.println("Java version: " + System.getProperty("java.version"));
System.out.println("Java vendor:  " + System.getProperty("java.vendor"));
System.out.println("OS arch:      " + System.getProperty("os.arch"));


## 5. Hello, LED Matrix — Your First Program

Before we touch any sensor, let's do the equivalent of `println("Hello world")` for the Sense HAT: light up the LED matrix.

Two objects are created every time you use the Sense HAT:

- `Context pi4j` — represents the connection to the Pi's hardware. Created once, closed at the end.
- `SenseHat senseHat` — the friendly wrapper around the HAT.

If everything is wired up correctly, running this cell should paint the whole matrix **green** for one second, then clear it.


In [ ]:
import com.pi4j.Pi4J;
import com.pi4j.context.Context;
import com.pi4j.drivers.hat.raspberry.SenseHat;

Context pi4j = Pi4J.newAutoContext();
SenseHat senseHat = new SenseHat(pi4j);

senseHat.fill(0x00FF00);   // green
Thread.sleep(1000);
senseHat.clear();

System.out.println("If you saw green LEDs, the Sense HAT is alive!");


## 6. Your First Sensor Reading

Now let's read something from the physical world. Warm the Pi up with your hand and watch the humidity change.

The Sense HAT actually has **two temperature sensors** — one inside the humidity chip (HTS221) and one inside the pressure chip (LPS25H). Neither is authoritative on its own; averaging them is a common trick.


In [ ]:
double humidity = senseHat.getHumidity();
double pressure = senseHat.getPressure();

double tempH = senseHat.getTemperatureFromHumidity();
double tempP = senseHat.getTemperatureFromPressure();
double temperature = (tempH + tempP) / 2.0;

System.out.printf("Humidity:    %.2f %%RH%n", humidity);
System.out.printf("Pressure:    %.2f hPa%n", pressure);
System.out.printf("Temperature: %.2f C  (H=%.2f, P=%.2f)%n", temperature, tempH, tempP);


### 6.1. Take a live reading every second for 10 seconds

Try breathing on the sensor while it runs — you should see humidity climb and temperature drift.


In [ ]:
for (int i = 1; i <= 10; i++) {
    double t = (senseHat.getTemperatureFromHumidity()
              + senseHat.getTemperatureFromPressure()) / 2.0;
    double h = senseHat.getHumidity();
    double p = senseHat.getPressure();

    System.out.printf("t=%2ds  temp=%.2f C  hum=%.2f %%  press=%.2f hPa%n",
        i, t, h, p);

    Thread.sleep(1000);
}


## 7. Motion Sensors — Tilt, Turn, and Compass

The LSM9DS1 is a **9-degrees-of-freedom** chip: 3 axes of acceleration, 3 axes of rotation, and 3 axes of magnetic field. Between them you can figure out how the Pi is tilted (pitch, roll, yaw), how it's spinning, and which way is north.

Tilt the Pi around while this cell runs — the numbers should follow.


In [ ]:
// Turn on all three IMU subsystems.
senseHat.setImuConfig(true, true, true);

for (int i = 0; i < 5; i++) {
    double[] accel   = senseHat.getAccelerometerRaw();  // g
    double[] gyro    = senseHat.getGyroscopeRaw();      // deg/s
    double[] orient  = senseHat.getOrientationDegrees();// pitch, roll, yaw
    double   heading = senseHat.getCompass();           // deg

    System.out.printf(
        "accel=[% .2f % .2f % .2f]  gyro=[% .2f % .2f % .2f]  "
        + "pitch=% .1f roll=% .1f yaw=% .1f  compass=% .1f%n",
        accel[0], accel[1], accel[2],
        gyro[0],  gyro[1],  gyro[2],
        orient[0], orient[1], orient[2],
        heading);

    Thread.sleep(500);
}


## 8. Drawing on the LED Matrix

Colours on the Sense HAT are 24-bit RGB, packed into an `int` as `0xRRGGBB`. So `0xFF0000` is red, `0x00FF00` is green, `0x0000FF` is blue. `0x000000` is off.

Individual pixels use `setPixel(x, y, color)`. The whole matrix can be updated in one go with a 2-D `int[8][8]` array — much faster than 64 individual calls.


In [ ]:
// Individual pixels: draw a red diagonal.
senseHat.clear();
for (int i = 0; i < 8; i++) {
    senseHat.setPixel(i, i, 0xFF0000);
    Thread.sleep(80);
}
Thread.sleep(500);

// A whole frame at once: a small heart.
// `_` is a reserved identifier in modern Java, so we use `o` for "off".
int R = 0xFF0033;
int o = 0x000000;
int[][] heart = {
    { o, R, R, o, o, R, R, o },
    { R, R, R, R, R, R, R, R },
    { R, R, R, R, R, R, R, R },
    { R, R, R, R, R, R, R, R },
    { o, R, R, R, R, R, R, o },
    { o, o, R, R, R, R, o, o },
    { o, o, o, R, R, o, o, o },
    { o, o, o, o, o, o, o, o },
};
senseHat.setPixels(heart);
Thread.sleep(1500);

senseHat.clear();


### 8.1. Scrolling text

`showMessage(text, scrollSpeedMillis, textColor, backColor)` scrolls a string across the matrix. Smaller `scrollSpeedMillis` = faster.


In [ ]:
senseHat.showMessage("Hello Sense HAT!", 60, 0xFFFFFF, 0x000033);
senseHat.clear();


### 8.2. Mini project — show the current temperature on the matrix

Combines what we've learned: read a sensor, format it as text, scroll it.


In [ ]:
double t = (senseHat.getTemperatureFromHumidity()
          + senseHat.getTemperatureFromPressure()) / 2.0;

String msg = String.format("%.1f C", t);
System.out.println("Scrolling: " + msg);

senseHat.showMessage(msg, 70, 0xFFAA00, 0x000000);
senseHat.clear();


## 9. Reading the Joystick

The tiny 5-way joystick reports events: **up**, **down**, **left**, **right**, and a **center click**. Each event has a key and an action (`PRESSED`, `RELEASED`, `HELD`).

The easiest pattern is to **poll** for events every so often.

**Move the joystick for 5 seconds** — the notebook will print each event and light up an arrow on the LED matrix pointing in the direction you pressed.


In [ ]:
import com.pi4j.drivers.hat.raspberry.SenseHat.Event;
import com.pi4j.drivers.hat.raspberry.SenseHat.Action;
import com.pi4j.io.gpio.digital.GameController;

long deadline = System.currentTimeMillis() + 5_000;
System.out.println("Move the joystick for 5 seconds...");

while (System.currentTimeMillis() < deadline) {
    for (Event e : senseHat.getEvents()) {
        if (e.action() != Action.PRESSED) continue;

        System.out.println("→ pressed " + e.key());
        senseHat.clear();

        GameController.Key k = e.key();
        if      (k == GameController.Key.UP)    senseHat.setPixel(4, 0, 0x00FF00);
        else if (k == GameController.Key.DOWN)  senseHat.setPixel(4, 7, 0x00FF00);
        else if (k == GameController.Key.LEFT)  senseHat.setPixel(0, 4, 0x00FF00);
        else if (k == GameController.Key.RIGHT) senseHat.setPixel(7, 4, 0x00FF00);
        else                                    senseHat.setPixel(4, 4, 0xFFFFFF);
    }
    Thread.sleep(50);
}

senseHat.clear();
System.out.println("Done.");


## 10. Cleanup

Always release the hardware when you're finished. `senseHat.close()` frees the LED matrix and joystick handles; `pi4j.shutdown()` releases the I2C descriptors. Failing to do this can leave the bus in a weird state and make the next run fail with cryptic errors.

We wrap each call in its own `try/catch` so that a failure in one shutdown step doesn't stop the other one.


In [ ]:
try {
    senseHat.clear();
    senseHat.close();
} catch (Exception e) {
    System.out.println("SenseHat close warning: " + e.getMessage());
}

try {
    pi4j.shutdown();
} catch (Exception e) {
    System.out.println("Pi4J shutdown warning: " + e.getMessage());
}

System.out.println("All done. Have fun!");


## Where to go from here

- **API tour.** For a systematic walkthrough of *every* method on the `SenseHat` class, open `01_SenseHAT_API_Tour.ipynb`. That notebook covers advanced pieces we didn't touch here: rotation and flips, per-letter rendering, the raw `GraphicsDisplay`, the light/colour sensor (v2 only), and the listener-based joystick pattern.
- **Real project.** The lakehouse notebooks (`02_SenseHAT_Lakehouse_Demo.ipynb` and `03_Athena_Glue_External_Tables_Demo.ipynb`) show one end-to-end use case: sensor → Spark → S3-compatible storage → Athena queries.
- **Pi4J docs.** Reference for the underlying library: [https://pi4j.com/](https://pi4j.com/)
- **Sense HAT driver source.** [pi4j-drivers SenseHat.java](https://github.com/igfasouza/pi4j-drivers/blob/igfasouza/src/main/java/com/pi4j/drivers/hat/raspberry/SenseHat.java)

## Troubleshooting

**`Cannot open /dev/i2c-1`** — I2C not enabled on the Pi, or the container doesn't have the device mapped.

**`java.lang.UnsatisfiedLinkError` on Pi4J FFM** — running on Java < 25 (FFM APIs are finalized in 25) or on a 32-bit JVM. Check the Java version cell in section 4.

**LED matrix stays dark** — the ATtiny at address `0x46` shows up in `i2cdetect` but stays off if the container is missing `/dev/fb0` or lacks `privileged: true`.

**Second run of the notebook fails** — you probably didn't call `senseHat.close()` / `pi4j.shutdown()` in the previous run. Restart the kernel and try again.
